# 02_pretrain.ipynb — SentinelFatal2 Pre-training Pipeline

**Agent 3 — Pre-training Pipeline (שלב 3)**  
Source: `src/train/pretrain.py`, `src/data/masking.py`, `src/data/dataset.py`  
SSOT: `docs/work_plan.md`, Part ה.4 + חלק ו שלב 3

---

## Notebook structure

| Cell | Purpose |
|------|----------|
| 1 | Setup: paths, seed, sys.path (+ Colab Drive mount) |
| 2 | GPU check |
| 3 | Load config + build model with PretrainingHead |
| 4 | **V3.1** Masking stability test — 10,000 seeds |
| 5 | **V3.2 + V3.3** Validate masking guarantees + loss on FHR only |
| 6 | **V3.4 + V3.5 + V3.6** Full pre-training (or dry-run) |
| 7 | Plot loss curves from logs/pretrain_loss.csv |

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os, random
import numpy as np

# ── Colab: mount Drive and cd to project ──────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/SentinelFatal2'   # adjust if needed
    os.chdir(PROJECT_ROOT)
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
else:
    # Local / VS Code
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'Project root : {os.getcwd()}')
print(f'Python       : {sys.version}')

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────────
import torch
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    DEVICE = torch.device('cpu')
    print('⚠ No GPU found — running on CPU (training will be slow)')

print(f'torch {torch.__version__}  |  device: {DEVICE}')

In [ ]:
# ── Cell 3: Load config + build model ─────────────────────────────────────────
from src.model.patchtst import PatchTST, load_config
from src.model.heads import PretrainingHead

CONFIG_PATH = 'config/train_config.yaml'
cfg = load_config(CONFIG_PATH)

model = PatchTST(cfg)
model.replace_head(PretrainingHead(
    d_model   = cfg['model']['d_model'],
    patch_len = cfg['data']['patch_len'],
))
model = model.to(DEVICE)

print(model)
print(f'Encoder params: {model.n_encoder_params:,}')
print(f'Head          : {type(model.head).__name__}')

In [ ]:
# ── Cell 4: V3.1 — Masking stability test (10,000 seeds) ─────────────────────
from src.data.masking import apply_masking

N_SEEDS = 10_000
MASK_RATIO     = cfg['pretrain']['mask_ratio']
MIN_GROUP_SIZE = cfg['pretrain']['min_group_size']
MAX_GROUP_SIZE = cfg['pretrain']['max_group_size']
N_PATCHES      = cfg['data']['n_patches']   # 73
PATCH_LEN      = cfg['data']['patch_len']   # 48
TARGET_MASKED  = round(MASK_RATIO * N_PATCHES)  # 29

print(f'Testing {N_SEEDS:,} seeds  |  n_patches={N_PATCHES}, '
      f'mask_ratio={MASK_RATIO}, target_masked={TARGET_MASKED}')

dummy_patches = np.random.rand(N_PATCHES, PATCH_LEN).astype(np.float32)
failures = []

for seed in range(N_SEEDS):
    random.seed(seed)
    np.random.seed(seed)
    patches_copy = dummy_patches.copy()
    try:
        _, idx = apply_masking(
            patches_copy,
            mask_ratio=MASK_RATIO,
            min_group_size=MIN_GROUP_SIZE,
            max_group_size=MAX_GROUP_SIZE,
        )
        assert idx[0] >= 1,          f'seed={seed}: idx[0]={idx[0]} < 1 (boundary)'
        assert idx[-1] <= N_PATCHES - 2, f'seed={seed}: idx[-1]={idx[-1]} >= {N_PATCHES-1} (boundary)'
        assert len(idx) == TARGET_MASKED, f'seed={seed}: len(idx)={len(idx)} != {TARGET_MASKED}'
    except Exception as e:
        failures.append((seed, str(e)))

if failures:
    print(f'✗ {len(failures)} failures (first 5):')
    for s, e in failures[:5]:
        print(f'  seed={s}: {e}')
else:
    print(f'✓ V3.1 PASS — {N_SEEDS:,} seeds passed, masking is stable')

In [ ]:
# ── Cell 5: V3.2 + V3.3 — Masking guarantees + loss on FHR only ──────────────
import torch.nn.functional as F
from src.train.pretrain import pretrain_step, generate_mask_indices

BATCH = 4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

x = torch.randn(BATCH, 2, 1800, device=DEVICE)

# V3.2 — masking guarantees
mask_indices = generate_mask_indices(
    n_patches=N_PATCHES, mask_ratio=MASK_RATIO,
    min_group_size=MIN_GROUP_SIZE, max_group_size=MAX_GROUP_SIZE,
)
mask_arr = np.zeros(N_PATCHES, dtype=bool)
mask_arr[mask_indices] = True

# Boundary preservation
assert not mask_arr[0],  'FAIL V3.2: first patch is masked'
assert not mask_arr[-1], 'FAIL V3.2: last patch is masked'

# Correct count
assert len(mask_indices) == TARGET_MASKED, \
    f'FAIL V3.2: got {len(mask_indices)} masked patches, expected {TARGET_MASKED}'

# Every contiguous group ≥ min_group_size
diff = np.diff(np.concatenate([[0], mask_arr.astype(int), [0]]))
run_starts = np.where(diff == 1)[0]
run_ends   = np.where(diff == -1)[0]
for s, e in zip(run_starts, run_ends):
    assert (e - s) >= MIN_GROUP_SIZE, \
        f'FAIL V3.2: group of length {e-s} < {MIN_GROUP_SIZE} at position {s}'

print(f'✓ V3.2 PASS — boundary={not mask_arr[0] and not mask_arr[-1]}, '
      f'n_masked={len(mask_indices)}, n_groups={len(run_starts)}, '
      f'group_sizes={[int(e-s) for s,e in zip(run_starts, run_ends)][:8]}...')

# V3.3 — loss computed on FHR masked patches ONLY (not UC)
model.eval()
with torch.no_grad():
    pred, target = pretrain_step(model, x, mask_indices)

assert pred.shape   == (BATCH, TARGET_MASKED, PATCH_LEN), \
    f'FAIL V3.3: pred shape {pred.shape}'
assert target.shape == (BATCH, TARGET_MASKED, PATCH_LEN), \
    f'FAIL V3.3: target shape {target.shape}'

loss = F.mse_loss(pred, target)
assert loss.item() > 0, 'FAIL V3.3: loss is zero (suspicious)'
assert not torch.isnan(loss), 'FAIL V3.3: loss is NaN'

print(f'✓ V3.3 PASS — loss={loss.item():.6f} on ({BATCH}, {TARGET_MASKED}, {PATCH_LEN}) '
      f'masked FHR patches only')
model.train()

In [ ]:
# ── Cell 6: V3.4 + V3.5 + V3.6 — Full Pre-training ───────────────────────────
#
# Set DRY_RUN = True (default) to run 2 batches only (CPU verification).
# Set DRY_RUN = False + DEVICE = cuda for the full Colab training run.
#
import subprocess, sys

DRY_RUN    = True        # ← change to False for full training on Colab
BATCH_SIZE = 4 if DRY_RUN else cfg['pretrain']['batch_size']
MAX_BATCHES = 2 if DRY_RUN else 0

cmd = [
    sys.executable, 'src/train/pretrain.py',
    '--config',      'config/train_config.yaml',
    '--device',      str(DEVICE),
    '--batch-size',  str(BATCH_SIZE),
    '--max-batches', str(MAX_BATCHES),
]

if DRY_RUN:
    print('[dry-run] Command:', ' '.join(cmd))
    print('[dry-run] Running 2 batches on', DEVICE, '...')

result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('\n✗ Pre-training script exited with code', result.returncode)
else:
    if DRY_RUN:
        print('\n✓ V3.4 PASS — forward→backward→step completed without errors')
    else:
        print('\n✓ Full pre-training complete')

In [ ]:
# ── Cell 6b: Verify artifacts (V3.5 checkpoint, V3.6 early stopping log) ─────
import os

# V3.5 — checkpoint saved
ckpt_path = 'checkpoints/pretrain/epoch_000.pt'
best_path = 'checkpoints/pretrain/best_pretrain.pt'
assert os.path.exists(ckpt_path), f'FAIL V3.5: checkpoint not found at {ckpt_path}'
assert os.path.exists(best_path), f'FAIL V3.5: best checkpoint not found at {best_path}'

# Verify checkpoint loads correctly
loaded = torch.load(best_path, map_location='cpu', weights_only=True)
assert isinstance(loaded, dict), 'FAIL V3.5: checkpoint is not a state dict'
print(f'✓ V3.5 PASS — checkpoints saved ({len(loaded)} param tensors)')

# V3.6 — loss CSV logged
log_path = 'logs/pretrain_loss.csv'
assert os.path.exists(log_path), f'FAIL V3.6: log not found at {log_path}'
import csv
with open(log_path) as f:
    rows = list(csv.DictReader(f))
assert len(rows) >= 1, 'FAIL V3.6: no rows in loss CSV'
assert 'train_loss' in rows[0] and 'val_loss' in rows[0], 'FAIL V3.6: missing columns'
print(f'✓ V3.6 PASS — loss CSV has {len(rows)} row(s): {rows[0]}')

In [ ]:
# ── Cell 7: Plot loss curves ─────────────────────────────────────────────────
import csv
import matplotlib.pyplot as plt

log_path = 'logs/pretrain_loss.csv'
if not os.path.exists(log_path):
    print(f'Log file not found: {log_path} — run Cell 6 first.')
else:
    with open(log_path) as f:
        rows = list(csv.DictReader(f))

    if len(rows) < 2:
        print(f'Only {len(rows)} epoch(s) logged — not enough for a meaningful plot.')
        print('Dry-run data:', rows)
    else:
        epochs      = [int(r['epoch'])        for r in rows]
        train_loss  = [float(r['train_loss']) for r in rows]
        val_loss    = [float(r['val_loss'])   for r in rows]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(epochs, train_loss, label='Train MSE', marker='.')
        ax.plot(epochs, val_loss,   label='Val MSE',   marker='.', linestyle='--')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Reconstruction MSE Loss')
        ax.set_title('Pre-training Loss Curves')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('logs/pretrain_loss_curve.png', dpi=120)
        plt.show()
        print('Saved → logs/pretrain_loss_curve.png')